# Distributed Spam Detection

For presentational purposes I assume Ray is running on port 8080.

In [4]:
import ray

In [5]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-05-31 13:02:12,480	INFO worker.py:2012 -- Started a local Ray instance.
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1



## Data loading and preprocessing

In [34]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-05-31 13:34:11,263	INFO logging.py:416 -- Registered dataset logger for dataset dataset_70_0
2026-05-31 13:34:11,266	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_70_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_13-02-08_943553_14132\logs\ray-data
2026-05-31 13:34:11,266	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_70_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-05-31 13:34:11,281	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_70_0 =======
2026-05-31 13:34:11,284	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 13:34:11,284	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/370.0MiB object store
2026-05-31 13:34:11,285	INFO logging_progress.py:181 -- 
2026-05-31 13:34:11,285	INFO logging_progress.py:231 -- limit=5: 0/1
2026-05-31 13:34:11,285	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [35]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-05-31 13:34:14,240	INFO logging.py:416 -- Registered dataset logger for dataset dataset_73_0
2026-05-31 13:34:14,242	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_73_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_13-02-08_943553_14132\logs\ray-data
2026-05-31 13:34:14,243	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_73_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-05-31 13:34:14,250	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_73_0 =======
2026-05-31 13:34:14,252	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 13:34:14,252	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/370.0MiB object store
2026-05-31 13:34:14,252	INFO logging_progress.py:181 -- 
2026-05-31 13:34:14,253	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-05-31 13:34:14,253	INFO logging_prog

Train set:
{'Message ID': 4871, 'Subject': 'today is a day to remember', 'Message': 'congradulations ,\nout of tens of thousands you are lucky enough to recieve interest rates starting at 2 . 95 %\nregardless of your past credit history . this is the break that you have been looking for and we are here\nto help you save hundreds every month . in order to lock in your record low rates we ask you\nto please fill out our 30 second form that is completly\nfree of charge . http : / / www . crrefi . net / ? id = t 52\nsincerly ,\nmichael smith ceo lowrate lenders corp .\nsophocles vibrate swami mountainside\ndietrich sermon munich mae\n', 'Spam/Ham': 1, 'Date': '2005-04-05', 'Text': 'today is a day to remember congradulations ,\nout of tens of thousands you are lucky enough to recieve interest rates starting at 2 . 95 %\nregardless of your past credit history . this is the break that you have been looking for and we are here\nto help you save hundreds every month . in order to lock in your r

2026-05-31 13:34:14,597	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_78_0 execution finished in 0.15 seconds


{'Message ID': 14284, 'Subject': 'proposed employee letter on retention payments', 'Message': 'louise ,\nattached is the draft letter that mark palmer discussed with you . your review and comments , particularly related to the second paragraph on the dynegy bonuses , would be appreciated .\nthanks ,\njohn', 'Spam/Ham': 0, 'Date': '2001-12-06', 'Text': 'proposed employee letter on retention payments louise ,\nattached is the draft letter that mark palmer discussed with you . your review and comments , particularly related to the second paragraph on the dynegy bonuses , would be appreciated .\nthanks ,\njohn'}
